# LeetCode 2373 — Largest Local Values in a Matrix bằng Python

Link đề: https://leetcode.com/problems/largest-local-values-in-a-matrix/ · Độ khó: **Easy** · Chủ đề: Array, Matrix

Skeleton LeetCode:

```python
class Solution:
    def largestLocal(self, grid: List[List[int]]) -> List[List[int]]:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu bài toán **cửa sổ trượt 2 chiều** `3 x 3` trên ma trận: với mỗi vị trí, lấy **giá trị lớn nhất** trong 9 ô.
2. Tính đúng **kích thước kết quả** `(n - 2) x (n - 2)` và các **chỉ số** `i, j` hợp lệ.
3. Dùng lát cắt `row[j:j+3]`, `max` với biểu thức sinh, `zip` để ghép ba hàng/cột, và khái niệm **tách phép tính 2D thành hai lượt 1D** (max theo hàng rồi theo cột).
4. Viết 5 cách: bốn vòng `for` lồng, `max` với biểu thức sinh trên độ lệch, lát cắt, **hai lượt** (hàng rồi cột), và `zip`; đối chiếu bằng test ngẫu nhiên.
5. Tránh các lỗi: sai kích thước kết quả (`n` thay vì `n - 2`), chỉ số cửa sổ lệch (tâm thay vì góc trên-trái), dùng chung hàng (aliasing) khi tạo `[[0] * m] * k`, ghi đè sai chỗ.

## 2. Đề bài & ví dụ

Cho ma trận số nguyên `grid` kích thước `n x n`. Hãy tạo ma trận `maxLocal` kích thước `(n - 2) x (n - 2)` sao cho:

- `maxLocal[i][j]` là **giá trị lớn nhất** trong ma trận con `3 x 3` của `grid` có **tâm** ở hàng `i + 1`, cột `j + 1`. Tương đương: ma trận con có góc trên-trái ở `(i, j)` và phủ các hàng `i..i+2`, cột `j..j+2`.

Trả về `maxLocal`.

| Ví dụ | grid | Output |
|---|---|---|
| 1 | `[[9,9,8,1],[5,6,2,6],[8,2,6,4],[6,2,2,2]]` | `[[9,9],[8,6]]` |
| 2 | `[[1,1,1,1,1],[1,1,1,1,1],[1,1,2,1,1],[1,1,1,1,1],[1,1,1,1,1]]` | `[[2,2,2],[2,2,2],[2,2,2]]` |

Giải thích ví dụ 1: khối `3x3` ở góc trên-trái gồm `9,9,8,5,6,2,8,2,6` → max `9`; khối bên phải của nó `9,8,1,6,2,6,2,6,4` → `9`; v.v.

Constraints: `n == len(grid) == len(grid[i])`; `3 <= n <= 100`; `1 <= grid[i][j] <= 100`.

In [1]:
grid = [[9, 9, 8, 1], [5, 6, 2, 6], [8, 2, 6, 4], [6, 2, 2, 2]]
n = len(grid)
print("n =", n, "-> kết quả cỡ", n - 2, "x", n - 2)
for i in range(n - 2):
    for j in range(n - 2):
        block = [grid[r][c] for r in range(i, i + 3) for c in range(j, j + 3)]
        print(f"(i={i}, j={j}): khối={block} -> max={max(block)}")

n = 4 -> kết quả cỡ 2 x 2
(i=0, j=0): khối=[9, 9, 8, 5, 6, 2, 8, 2, 6] -> max=9
(i=0, j=1): khối=[9, 8, 1, 6, 2, 6, 2, 6, 4] -> max=9
(i=1, j=0): khối=[5, 6, 2, 8, 2, 6, 6, 2, 2] -> max=8
(i=1, j=1): khối=[6, 2, 6, 2, 6, 4, 2, 2, 2] -> max=6


## 3. Kiến thức nền

### 3.1. Ma trận là danh sách các danh sách

`grid[r][c]` là ô ở hàng `r`, cột `c`. `len(grid)` là số hàng, `len(grid[0])` là số cột. Ma trận vuông có hai số này bằng nhau.

In [2]:
grid = [[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 14, 15, 16]]
print(len(grid), len(grid[0]), grid[2][1], grid[1])

4 4 10 [5, 6, 7, 8]


### 3.2. Cửa sổ `3 x 3` và phạm vi chỉ số

Góc trên-trái ở `(i, j)` phủ hàng `i, i+1, i+2` và cột `j, j+1, j+2`. Để cả ba hàng/cột nằm trong `[0, n-1]` cần `i + 2 <= n - 1` ⇔ `i <= n - 3` ⇔ `i` chạy `0..n-3` tức **`range(n - 2)`**. Số cửa sổ mỗi chiều: `n - 2`.

In [3]:
for n in (3, 4, 5, 10):
    print(f"n={n}: i trong {list(range(n - 2))[:4]}{'...' if n > 6 else ''} -> {n - 2} cửa sổ mỗi chiều")

n=3: i trong [0] -> 1 cửa sổ mỗi chiều
n=4: i trong [0, 1] -> 2 cửa sổ mỗi chiều
n=5: i trong [0, 1, 2] -> 3 cửa sổ mỗi chiều
n=10: i trong [0, 1, 2, 3]... -> 8 cửa sổ mỗi chiều


### 3.3. `max` với biểu thức sinh và lát cắt

`max(iterable)` lấy lớn nhất. `row[j:j+3]` lấy ba phần tử liên tiếp từ cột `j`. Có thể lồng: `max(max(grid[r][j:j+3]) for r in range(i, i+3))`.

In [4]:
grid = [[9, 9, 8, 1], [5, 6, 2, 6], [8, 2, 6, 4], [6, 2, 2, 2]]
print(grid[0][0:3], max(grid[0][0:3]))
print(max(max(grid[r][0:3]) for r in range(0, 3)))
print(max(grid[r][c] for r in range(0, 3) for c in range(0, 3)))

[9, 9, 8] 9
9
9


### 3.4. `zip` để ghép nhiều dãy theo chỉ số

`zip(a, b, c)` ghép phần tử cùng chỉ số: `(a[0], b[0], c[0]), (a[1], b[1], c[1]), ...`. `zip(row, row[1:], row[2:])` cho **tất cả bộ ba liên tiếp** của `row` — chính xác là các cửa sổ độ rộng 3.

In [5]:
row = [9, 9, 8, 1]
print(list(zip(row, row[1:], row[2:])))
print([max(t) for t in zip(row, row[1:], row[2:])])

[(9, 9, 8), (9, 8, 1)]
[9, 9]


### 3.5. Tạo ma trận kết quả: tránh `[[0] * m] * k`

`[[0] * 3] * 2` tạo **hai tham chiếu đến cùng một hàng**; sửa hàng này đổi cả hàng kia. Hãy dùng list comprehension: `[[0] * 3 for _ in range(2)]`.

In [6]:
bad = [[0] * 3] * 2
bad[0][0] = 9
print(bad)                  # cả hai hàng đều đổi!
good = [[0] * 3 for _ in range(2)]
good[0][0] = 9
print(good)

[[9, 0, 0], [9, 0, 0]]
[[9, 0, 0], [0, 0, 0]]


### 3.6. Ý tưởng tách max 2D thành hai lượt 1D

`max` trên một khối `3 x 3` = `max` của (`max` mỗi hàng của khối). Vì vậy: **lượt 1** tính, cho mỗi hàng, `max` mọi cửa sổ ngang độ rộng 3 → ma trận `n x (n-2)`; **lượt 2** tính, trên mỗi cột của ma trận đó, `max` mọi cửa sổ dọc độ cao 3 → ma trận `(n-2) x (n-2)`. Mỗi ô chỉ cần 3 phép so sánh mỗi lượt (thay vì 9).

In [7]:
grid = [[9, 9, 8, 1], [5, 6, 2, 6], [8, 2, 6, 4], [6, 2, 2, 2]]
row_max = [[max(row[j:j + 3]) for j in range(len(row) - 2)] for row in grid]
print("sau lượt 1 (max ngang):", row_max)
final = [[max(row_max[i][j], row_max[i + 1][j], row_max[i + 2][j]) for j in range(len(row_max[0]))] for i in range(len(row_max) - 2)]
print("sau lượt 2 (max dọc)  :", final)

sau lượt 1 (max ngang): [[9, 9], [6, 6], [8, 6], [6, 2]]
sau lượt 2 (max dọc)  : [[9, 9], [8, 6]]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Đặt một "khung" `3 x 3` lên ma trận rồi **trượt** từng ô một, mỗi lần hỏi "số lớn nhất trong khung là gì?". Ma trận `4 x 4` có khung ở `2 x 2` vị trí (hai chiều ngang, hai chiều dọc).

```
9 9 8 1        khung(0,0): 9 9 8 / 5 6 2 / 8 2 6 -> 9
5 6 2 6        khung(0,1): 9 8 1 / 6 2 6 / 2 6 4 -> 9
8 2 6 4        khung(1,0): 5 6 2 / 8 2 6 / 6 2 2 -> 8
6 2 2 2        khung(1,1): 6 2 6 / 2 6 4 / 2 2 2 -> 6
```

Vậy kết quả `[[9, 9], [8, 6]]`. Quy tắc: **`result[i][j] = max của grid[i..i+2][j..j+2]`**.

In [8]:
grid = [[9, 9, 8, 1], [5, 6, 2, 6], [8, 2, 6, 4], [6, 2, 2, 2]]
result = []
for i in range(len(grid) - 2):
    row = []
    for j in range(len(grid) - 2):
        window_max = grid[i][j]
        for r in range(i, i + 3):
            for c in range(j, j + 3):
                window_max = max(window_max, grid[r][c])
        row.append(window_max)
    result.append(row)
print(result)

[[9, 9], [8, 6]]


## 5. Cách 1 — Brute Force / baseline: bốn vòng `for` lồng

Hai vòng ngoài chọn góc `(i, j)` của khung; hai vòng trong duyệt `9` ô của khung để tìm max. Trực tiếp theo định nghĩa.

In [9]:
def largest_local_loops(grid):
    n = len(grid)
    result = []
    for i in range(n - 2):
        row = []
        for j in range(n - 2):
            best = 0
            for r in range(i, i + 3):
                for c in range(j, j + 3):
                    if grid[r][c] > best:
                        best = grid[r][c]
            row.append(best)
        result.append(row)
    return result

print(largest_local_loops([[9, 9, 8, 1], [5, 6, 2, 6], [8, 2, 6, 4], [6, 2, 2, 2]]))

[[9, 9], [8, 6]]


In [10]:
from typing import List

class SolutionLoops:
    def largestLocal(self, grid: List[List[int]]) -> List[List[int]]:
        n = len(grid)
        result = []
        for i in range(n - 2):
            row = []
            for j in range(n - 2):
                best = 0
                for r in range(i, i + 3):
                    for c in range(j, j + 3):
                        if grid[r][c] > best:
                            best = grid[r][c]
                row.append(best)
            result.append(row)
        return result

print(SolutionLoops().largestLocal([[9, 9, 8, 1], [5, 6, 2, 6], [8, 2, 6, 4], [6, 2, 2, 2]]))
print(SolutionLoops().largestLocal([[1, 1, 1, 1, 1], [1, 1, 1, 1, 1], [1, 1, 2, 1, 1], [1, 1, 1, 1, 1], [1, 1, 1, 1, 1]]))

[[9, 9], [8, 6]]
[[2, 2, 2], [2, 2, 2], [2, 2, 2]]


**Độ phức tạp:** `n x n` ma trận, có `(n-2)²` khung, mỗi khung `9` ô → **Time `O(9·(n-2)²) = O(n²)`**; kết quả `(n-2)²` ô → **Space `O(n²)`** (cho kết quả; bộ nhớ phụ `O(1)`).

**Nút thắt:** mỗi ô được đọc tới `9` lần (nằm trong tối đa `9` khung). Hằng số 9 nhỏ nên bài vẫn đạt `O(n²)`; cải tiến chỉ để gọn hơn hoặc giảm hằng số.

## 6. Ý tưởng thuật toán / bước đột phá

- Công thức trực tiếp: `result[i][j] = max(grid[i+di][j+dj] for di in 0..2 for dj in 0..2)`.
- Chỉ số: `i`, `j` chạy trong `range(n - 2)`; kết quả `(n-2) x (n-2)`.
- **`max` tách được:** `max` của 9 ô = `max` của ba `max` hàng. Nên có thể tính `max` ngang trước (mỗi hàng cho `n - 2` giá trị), rồi `max` dọc trên kết quả — chỉ cần 3 phép so sánh mỗi lượt.
- `zip(row, row[1:], row[2:])` sinh sẵn các bộ ba ngang liên tiếp; `zip(r0, r1, r2)` cho từng cột của ba hàng liên tiếp — ghép hai ý trên thành một biểu thức ngắn gọn.

## 7. Các cách tiếp theo

### Cách 2 — `max` với biểu thức sinh trên độ lệch `di, dj`

In [11]:
from typing import List

class SolutionGenerator:
    def largestLocal(self, grid: List[List[int]]) -> List[List[int]]:
        n = len(grid)
        return [
            [max(grid[i + di][j + dj] for di in range(3) for dj in range(3)) for j in range(n - 2)]
            for i in range(n - 2)
        ]

print(SolutionGenerator().largestLocal([[9, 9, 8, 1], [5, 6, 2, 6], [8, 2, 6, 4], [6, 2, 2, 2]]))
print(SolutionGenerator().largestLocal([[1, 1, 1, 1, 1], [1, 1, 1, 1, 1], [1, 1, 2, 1, 1], [1, 1, 1, 1, 1], [1, 1, 1, 1, 1]]))

[[9, 9], [8, 6]]
[[2, 2, 2], [2, 2, 2], [2, 2, 2]]


**Độ phức tạp:** Time `O(n²)` (mỗi khung 9 ô), Space `O(n²)` cho kết quả.

### Cách 3 — Lát cắt hàng: `max(max(grid[r][j:j+3]) for r in ...)`

Mỗi hàng của khung lấy bằng lát cắt `grid[r][j:j+3]`, `max` hàng đó, rồi `max` của ba kết quả. Ngắn hơn Cách 2 và tận dụng lát cắt chạy bằng C.

In [12]:
from typing import List

class SolutionSlices:
    def largestLocal(self, grid: List[List[int]]) -> List[List[int]]:
        n = len(grid)
        result = []
        for i in range(n - 2):
            row = []
            for j in range(n - 2):
                row.append(max(max(grid[r][j:j + 3]) for r in range(i, i + 3)))
            result.append(row)
        return result

print(SolutionSlices().largestLocal([[9, 9, 8, 1], [5, 6, 2, 6], [8, 2, 6, 4], [6, 2, 2, 2]]))
print(SolutionSlices().largestLocal([[1, 1, 1, 1, 1], [1, 1, 1, 1, 1], [1, 1, 2, 1, 1], [1, 1, 1, 1, 1], [1, 1, 1, 1, 1]]))

[[9, 9], [8, 6]]
[[2, 2, 2], [2, 2, 2], [2, 2, 2]]


**Độ phức tạp:** Time `O(n²)`, Space `O(n²)` cho kết quả (lát cắt tạo bản sao nhỏ 3 phần tử).

### Cách 4 — Hai lượt: max ngang rồi max dọc

Lượt 1: `row_max[r][j] = max(grid[r][j], grid[r][j+1], grid[r][j+2])` (ma trận `n x (n-2)`). Lượt 2: `result[i][j] = max(row_max[i][j], row_max[i+1][j], row_max[i+2][j])`. Mỗi lượt chỉ 3 phần tử mỗi ô: ít phép so sánh hơn cách trực tiếp và là kỹ thuật chuẩn cho các bài "max/min/sum trong cửa sổ".

In [13]:
from typing import List

class SolutionTwoPass:
    def largestLocal(self, grid: List[List[int]]) -> List[List[int]]:
        n = len(grid)
        # Lượt 1: max ngang độ rộng 3
        row_max = [[0] * (n - 2) for _ in range(n)]
        for r in range(n):
            for j in range(n - 2):
                row_max[r][j] = max(grid[r][j], grid[r][j + 1], grid[r][j + 2])
        # Lượt 2: max dọc độ cao 3
        result = [[0] * (n - 2) for _ in range(n - 2)]
        for i in range(n - 2):
            for j in range(n - 2):
                result[i][j] = max(row_max[i][j], row_max[i + 1][j], row_max[i + 2][j])
        return result

print(SolutionTwoPass().largestLocal([[9, 9, 8, 1], [5, 6, 2, 6], [8, 2, 6, 4], [6, 2, 2, 2]]))
print(SolutionTwoPass().largestLocal([[1, 1, 1, 1, 1], [1, 1, 1, 1, 1], [1, 1, 2, 1, 1], [1, 1, 1, 1, 1], [1, 1, 1, 1, 1]]))

[[9, 9], [8, 6]]
[[2, 2, 2], [2, 2, 2], [2, 2, 2]]


**Độ phức tạp:** Time `O(n²)` (hằng số ~6 thay vì 9), Space `O(n²)` thêm cho `row_max`. Cách này mở rộng sang cửa sổ `k x k` với `O(n²·k)` thay vì `O(n²·k²)`.

### Cách 5 — `zip` thuần Python

`zip(row, row[1:], row[2:])` cho các bộ ba ngang → `max` mỗi bộ; rồi `zip(r0, r1, r2)` trên ba hàng liên tiếp của kết quả lượt 1 cho các cột → `max`. Không có chỉ số, không có vòng `for` thủ công.

In [14]:
from typing import List

class SolutionZip:
    def largestLocal(self, grid: List[List[int]]) -> List[List[int]]:
        horizontal = [[max(t) for t in zip(row, row[1:], row[2:])] for row in grid]
        return [
            [max(col) for col in zip(top, mid, bottom)]
            for top, mid, bottom in zip(horizontal, horizontal[1:], horizontal[2:])
        ]

print(SolutionZip().largestLocal([[9, 9, 8, 1], [5, 6, 2, 6], [8, 2, 6, 4], [6, 2, 2, 2]]))
print(SolutionZip().largestLocal([[1, 1, 1, 1, 1], [1, 1, 1, 1, 1], [1, 1, 2, 1, 1], [1, 1, 1, 1, 1], [1, 1, 1, 1, 1]]))

[[9, 9], [8, 6]]
[[2, 2, 2], [2, 2, 2], [2, 2, 2]]


**Độ phức tạp:** Time `O(n²)`, Space `O(n²)` (ma trận `horizontal` và kết quả). Ngắn nhất nhưng hơi "ma thuật" với người mới; hãy đọc kỹ cách `zip` ghép các cửa sổ.

## 8. TRACE từng bước

Trace Cách 4 (hai lượt) với `grid = [[9,9,8,1],[5,6,2,6],[8,2,6,4],[6,2,2,2]]`.

**Lượt 1 (max ngang độ rộng 3):**

| hàng | các cửa sổ ngang | `row_max` |
|---|---|---|
| 0 | `[9,9,8]`, `[9,8,1]` | `[9, 9]` |
| 1 | `[5,6,2]`, `[6,2,6]` | `[6, 6]` |
| 2 | `[8,2,6]`, `[2,6,4]` | `[8, 6]` |
| 3 | `[6,2,2]`, `[2,2,2]` | `[6, 2]` |

**Lượt 2 (max dọc độ cao 3):** cột 0: `max(9,6,8)=9`, `max(6,8,6)=8`; cột 1: `max(9,6,6)=9`, `max(6,6,2)=6`. Kết quả `[[9,9],[8,6]]`.

In [15]:
def trace_two_pass(grid):
    n = len(grid)
    print(f"n={n}")
    row_max = []
    print("Lượt 1 (max ngang):")
    for r in range(n):
        values = [max(grid[r][j], grid[r][j + 1], grid[r][j + 2]) for j in range(n - 2)]
        row_max.append(values)
        windows = [grid[r][j:j + 3] for j in range(n - 2)]
        print(f"  hàng {r}: cửa sổ={windows} -> {values}")
    print("Lượt 2 (max dọc):")
    result = []
    for i in range(n - 2):
        row = []
        for j in range(n - 2):
            col = [row_max[i][j], row_max[i + 1][j], row_max[i + 2][j]]
            row.append(max(col))
            print(f"  (i={i}, j={j}): cột={col} -> {max(col)}")
        result.append(row)
    print("kết quả:", result)
    return result

trace_two_pass([[9, 9, 8, 1], [5, 6, 2, 6], [8, 2, 6, 4], [6, 2, 2, 2]])

n=4
Lượt 1 (max ngang):
  hàng 0: cửa sổ=[[9, 9, 8], [9, 8, 1]] -> [9, 9]
  hàng 1: cửa sổ=[[5, 6, 2], [6, 2, 6]] -> [6, 6]
  hàng 2: cửa sổ=[[8, 2, 6], [2, 6, 4]] -> [8, 6]
  hàng 3: cửa sổ=[[6, 2, 2], [2, 2, 2]] -> [6, 2]
Lượt 2 (max dọc):
  (i=0, j=0): cột=[9, 6, 8] -> 9
  (i=0, j=1): cột=[9, 6, 6] -> 9
  (i=1, j=0): cột=[6, 8, 6] -> 8
  (i=1, j=1): cột=[6, 6, 2] -> 6
kết quả: [[9, 9], [8, 6]]


[[9, 9], [8, 6]]

## 9. Code nộp LeetCode

In [16]:
from typing import List

class Solution:
    def largestLocal(self, grid: List[List[int]]) -> List[List[int]]:
        n = len(grid)
        return [
            [max(grid[r][c] for r in range(i, i + 3) for c in range(j, j + 3)) for j in range(n - 2)]
            for i in range(n - 2)
        ]

print(Solution().largestLocal([[9, 9, 8, 1], [5, 6, 2, 6], [8, 2, 6, 4], [6, 2, 2, 2]]))
print(Solution().largestLocal([[1, 1, 1, 1, 1], [1, 1, 1, 1, 1], [1, 1, 2, 1, 1], [1, 1, 1, 1, 1], [1, 1, 1, 1, 1]]))

[[9, 9], [8, 6]]
[[2, 2, 2], [2, 2, 2], [2, 2, 2]]


## 10. Vì sao thuật toán đúng

- **Kích thước đúng:** khung `3 x 3` có góc trên-trái ở `(i, j)` hợp lệ ⇔ `0 <= i, j <= n - 3`, tức `n - 2` giá trị mỗi chiều. Ma trận kết quả cỡ `(n-2) x (n-2)` khớp đề, và `result[i][j]` ứng với khối có **tâm** `(i+1, j+1)`.
- **Định nghĩa:** `result[i][j]` đúng bằng `max` của 9 ô `grid[r][c]` với `r ∈ {i, i+1, i+2}`, `c ∈ {j, j+1, j+2}`; mọi cách đều tính chính đại lượng đó.
- **Hai lượt đúng:** `max` có tính kết hợp/giao hoán: `max` trên tập 9 ô = `max` trên ba hàng của (`max` trên ba cột của hàng đó). Lượt 1 tính `max` của ba ô liên tiếp cùng hàng; lượt 2 lấy `max` của ba giá trị lượt 1 ở ba hàng liên tiếp cùng cột — đúng các ô `3 x 3` đó.
- **`zip` đúng:** `zip(row, row[1:], row[2:])` dừng ở dãy ngắn nhất (`row[2:]` có `n - 2` phần tử) nên cho đúng `n - 2` bộ ba. Tương tự cho các hàng.
- **Không ghi đè dữ liệu gốc:** mọi cách tạo ma trận mới; `grid` không bị sửa.

## 11. Phân tích Time / Space Complexity

`n x n` ma trận, `n ≤ 100`; `(n-2)²` khung.

| Cách | Time | Space (phụ, không tính kết quả) |
|---|---|---|
| 1. Bốn vòng `for` | `O(9n²)` = `O(n²)` | `O(1)` |
| 2. `max` biểu thức sinh | `O(n²)` | `O(1)` |
| 3. Lát cắt hàng | `O(n²)` | `O(1)` (lát cắt 3 phần tử) |
| 4. Hai lượt | `O(n²)` | `O(n²)` (`row_max`) |
| 5. `zip` | `O(n²)` | `O(n²)` (`horizontal`) |

Kết quả luôn `O(n²)` bộ nhớ. Mọi cách đều `O(n²)` về bậc, tối ưu vì phải đọc hết `n²` ô. Với cửa sổ `k x k` tổng quát, cách hai lượt cho `O(n²·k)`, còn trực tiếp là `O(n²·k²)`.

## 12. Bộ test

Ví dụ đề, edge case (`n = 3` nhỏ nhất cho kết quả `1 x 1`, `n = 4`, tất cả bằng nhau, cực đại ở góc/tâm, ma trận tăng dần) và test ngẫu nhiên đối chiếu **tất cả cách**, kèm kiểm tra rằng `grid` không bị thay đổi.

In [17]:
import copy, random

solvers = [SolutionLoops(), SolutionGenerator(), SolutionSlices(), SolutionTwoPass(), SolutionZip(), Solution()]

fixed = [
    ([[9, 9, 8, 1], [5, 6, 2, 6], [8, 2, 6, 4], [6, 2, 2, 2]], [[9, 9], [8, 6]]),
    ([[1, 1, 1, 1, 1], [1, 1, 1, 1, 1], [1, 1, 2, 1, 1], [1, 1, 1, 1, 1], [1, 1, 1, 1, 1]], [[2, 2, 2], [2, 2, 2], [2, 2, 2]]),
    ([[1, 2, 3], [4, 5, 6], [7, 8, 9]], [[9]]),                           # n = 3 -> 1 x 1
    ([[9, 1, 1], [1, 1, 1], [1, 1, 1]], [[9]]),                           # cực đại ở góc
    ([[1, 1, 1], [1, 9, 1], [1, 1, 1]], [[9]]),                           # cực đại ở tâm
    ([[1, 1, 1], [1, 1, 1], [1, 1, 9]], [[9]]),                           # cực đại ở góc dưới
    ([[5] * 4 for _ in range(4)], [[5, 5], [5, 5]]),                      # tất cả bằng nhau
    ([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 14, 15, 16]], [[11, 12], [15, 16]]),
    ([[100, 1, 1, 1], [1, 1, 1, 1], [1, 1, 1, 1], [1, 1, 1, 100]], [[100, 1], [1, 100]]),
]
for grid, expected in fixed:
    for sol in solvers:
        original = copy.deepcopy(grid)
        got = sol.largestLocal(grid)
        assert got == expected, (type(sol).__name__, grid, got, expected)
        assert grid == original, ("grid bị sửa", type(sol).__name__)
print("test cố định: OK")

def reference(grid):
    n = len(grid)
    out = []
    for i in range(n - 2):
        row = []
        for j in range(n - 2):
            block = []
            for r in range(i, i + 3):
                for c in range(j, j + 3):
                    block.append(grid[r][c])
            row.append(sorted(block)[-1])
        out.append(row)
    return out

random.seed(2373)
for _ in range(500):
    n = random.randint(3, 9)
    hi = random.choice([2, 10, 100])
    grid = [[random.randint(1, hi) for _ in range(n)] for _ in range(n)]
    expected = reference(grid)
    assert len(expected) == n - 2 and all(len(r) == n - 2 for r in expected)
    for sol in solvers:
        assert sol.largestLocal(grid) == expected, (type(sol).__name__, grid)
print("500 test ngẫu nhiên: OK")

big = [[random.randint(1, 100) for _ in range(100)] for _ in range(100)]
big_expected = reference(big)
for sol in solvers:
    assert sol.largestLocal(big) == big_expected, type(sol).__name__
print("ma trận 100 x 100: OK, kết quả cỡ", len(big_expected), "x", len(big_expected[0]))

test cố định: OK


500 test ngẫu nhiên: OK


ma trận 100 x 100: OK, kết quả cỡ

 98 x 98


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Kết quả cỡ `n x n` thay vì `(n-2) x (n-2)`

Khung `3 x 3` không "vừa" ở rìa; chỉ có `n - 2` vị trí mỗi chiều.

In [18]:
def wrong1(grid):
    n = len(grid)
    out = []
    for i in range(n):
        row = []
        for j in range(n):
            vals = [grid[r][c] for r in range(i, min(i + 3, n)) for c in range(j, min(j + 3, n))]
            row.append(max(vals))
        out.append(row)
    return out

g = [[9, 9, 8, 1], [5, 6, 2, 6], [8, 2, 6, 4], [6, 2, 2, 2]]
print("sai :", wrong1(g), "(cỡ 4x4; đáng lẽ 2x2)")
print("đúng:", Solution().largestLocal(g))

sai : [[9, 9, 8, 6], [8, 6, 6, 6], [8, 6, 6, 4], [6, 2, 2, 2]] (cỡ 4x4; đáng lẽ 2x2)
đúng: [[9, 9], [8, 6]]


### Lỗi 2: Lấy `range(n - 3)` (thiếu một vị trí) hoặc `range(n)` (vượt biên)

Số vị trí đúng là `n - 2`. `range(n - 3)` bỏ sót hàng/cột cuối; `range(n)` gây `IndexError`.

In [19]:
def wrong2(grid):
    n = len(grid)
    return [[max(grid[r][c] for r in range(i, i + 3) for c in range(j, j + 3)) for j in range(n - 3)] for i in range(n - 3)]

g = [[9, 9, 8, 1], [5, 6, 2, 6], [8, 2, 6, 4], [6, 2, 2, 2]]
print("sai :", wrong2(g), "(n - 3 = 1: chỉ một khung; đáng lẽ [[9,9],[8,6]])")
print("đúng:", Solution().largestLocal(g))

sai : [[9]] (n - 3 = 1: chỉ một khung; đáng lẽ [[9,9],[8,6]])
đúng: [[9, 9], [8, 6]]


### Lỗi 3: Lấy khung `3 x 3` lệch (chỉ số `i - 1..i + 1`)

Nếu nghĩ `(i, j)` là **tâm**, khung là `i-1..i+1`; nhưng khi lặp `i` từ `0` sẽ có chỉ số `-1` (lấy ô cuối cùng nhờ chỉ số âm của Python!), tạo kết quả sai âm thầm.

In [20]:
def wrong3(grid):
    n = len(grid)
    out = []
    for i in range(n - 2):
        row = []
        for j in range(n - 2):
            row.append(max(grid[r][c] for r in range(i - 1, i + 2) for c in range(j - 1, j + 2)))   # lệch
        out.append(row)
    return out

g = [[9, 9, 8, 1], [5, 6, 2, 6], [8, 2, 6, 4], [6, 2, 2, 2]]
print("sai :", wrong3(g), "(đáng lẽ [[9,9],[8,6]]; chỉ số -1 quấn về cuối)")
print("đúng:", Solution().largestLocal(g))

sai : [[9, 9], [9, 9]] (đáng lẽ [[9,9],[8,6]]; chỉ số -1 quấn về cuối)
đúng: [[9, 9], [8, 6]]


### Lỗi 4: Tạo ma trận bằng `[[0] * m] * k` (aliasing)

Mọi hàng là **cùng một đối tượng**; ghi một hàng ghi tất cả.

In [21]:
def wrong4(grid):
    n = len(grid)
    result = [[0] * (n - 2)] * (n - 2)               # các hàng là một đối tượng duy nhất
    for i in range(n - 2):
        for j in range(n - 2):
            result[i][j] = max(grid[r][c] for r in range(i, i + 3) for c in range(j, j + 3))
    return result

g = [[9, 9, 8, 1], [5, 6, 2, 6], [8, 2, 6, 4], [6, 2, 2, 2]]
print("sai :", wrong4(g), "(hàng cuối ghi đè hàng đầu; đáng lẽ [[9,9],[8,6]])")
print("đúng:", Solution().largestLocal(g))

sai : [[8, 6], [8, 6]] (hàng cuối ghi đè hàng đầu; đáng lẽ [[9,9],[8,6]])
đúng: [[9, 9], [8, 6]]


### Lỗi 5: Khởi tạo `best = 0` khi giá trị có thể âm

Ở đề này `grid[i][j] ≥ 1` nên `best = 0` vẫn đúng; nhưng với ma trận có số âm, `best = 0` sẽ sai. Hãy khởi tạo `best = grid[i][j]` hoặc dùng `max(...)` trực tiếp để mã bền vững.

In [22]:
def wrong5(grid):
    n = len(grid)
    out = []
    for i in range(n - 2):
        row = []
        for j in range(n - 2):
            best = 0                                      # lỗi tiềm ẩn nếu có số âm
            for r in range(i, i + 3):
                for c in range(j, j + 3):
                    best = max(best, grid[r][c])
            row.append(best)
        out.append(row)
    return out

neg = [[-5, -4, -3], [-2, -9, -1], [-7, -8, -6]]
print("sai :", wrong5(neg), "(trả 0; đáng lẽ -1)")
print("đúng:", Solution().largestLocal(neg))

sai : [[0]] (trả 0; đáng lẽ -1)
đúng: [[-1]]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Cửa sổ `k x k` trượt trên ma trận" → số vị trí mỗi chiều là `n - k + 1` (ở đây `n - 2`).
- `result[i][j]` ↔ khối có góc trên-trái `(i, j)`: dùng `range(i, i + 3)`, `range(j, j + 3)`, **không** dùng chỉ số âm.
- Tách phép 2D thành hai lượt 1D (`max` ngang rồi dọc) → giảm hằng số, mở rộng cho cửa sổ lớn (hoặc dùng hàng đợi đơn điệu cho `O(n²)` không phụ thuộc `k`).
- `zip(row, row[1:], row[2:])` là mẹo gọn cho "mọi bộ ba liên tiếp".
- Tạo ma trận mới bằng list comprehension, không dùng `[[0]*m]*k`.
- Cùng họ: LeetCode 661 Image Smoother, 1314 Matrix Block Sum, 363, 1738.

## 15. Biến thể của bài

1. Cửa sổ `k x k` tổng quát; kết quả cỡ `(n-k+1)²`.
2. LeetCode 661 Image Smoother: **trung bình** (làm tròn xuống) của khối `3 x 3`, xử lý khối ở rìa.
3. LeetCode 1314 Matrix Block Sum: **tổng** khối bằng tổng tiền tố (prefix sum 2D).
4. Tìm khối `3 x 3` có **tổng lớn nhất** và vị trí của nó.
5. Thay `max` bằng `min`, hoặc kết hợp `max - min` (độ chênh trong khối).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space phụ | Khi nào dùng |
|---|---|---|---|---|
| 1. Bốn vòng `for` | định nghĩa trực tiếp | `O(9n²)` | `O(1)` | hiểu bài, dễ debug |
| 2. `max` biểu thức sinh | gọn trong một dòng | `O(n²)` | `O(1)` | **khuyến nghị khi nộp** |
| 3. Lát cắt hàng | `max(grid[r][j:j+3])` | `O(n²)` | `O(1)` | ngắn, dùng lát cắt |
| 4. Hai lượt | max ngang rồi dọc | `O(n²)` (hằng số nhỏ) | `O(n²)` | cửa sổ lớn `k`, tối ưu |
| 5. `zip` | bộ ba ngang & dọc | `O(n²)` | `O(n²)` | thích Python thuần, không chỉ số |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** `print(list(zip([1,2,3,4], [1,2,3,4][1:], [1,2,3,4][2:])), max(max([1,5,2]), max([4,3])))` in ra gì?

**Bài 2 (tính tay).** `grid = [[1,2,3],[4,9,6],[7,8,5]]`. `largestLocal` trả về gì? Kích thước bao nhiêu?

**Bài 3 (điền TODO).** Hoàn thành hàm **min** cục bộ `3 x 3` (thay `max` bằng `min`).

In [23]:
def smallest_local(grid):
    # TODO: giống largestLocal nhưng lấy min của mỗi khối 3 x 3
    pass

# chưa gọi hàm ở đây để notebook chạy không lỗi

**Bài 4 (tự viết lại).** Viết `largest_local_k(grid, k)` cho cửa sổ `k x k` bất kỳ (`k ≤ n`), kết quả cỡ `(n-k+1) x (n-k+1)`.

In [24]:
def largest_local_k(grid, k):
    # TODO: tổng quát hoá chỉ số 3 thành k
    pass

**Bài 5 (mini challenge).** Viết `smoother(grid)` (LeetCode 661 đơn giản): với mỗi ô, trung bình (làm tròn xuống) các ô trong khối `3 x 3` **nằm trong ma trận** (khối ở rìa chỉ có 4–6 ô).

In [25]:
def smoother(grid):
    # TODO: với mỗi (i, j) lấy các ô r in [i-1, i+1], c in [j-1, j+1] còn trong biên, tổng // số ô
    pass

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `zip` cho `[(1,2,3), (2,3,4)]`; `max(5, 4) = 5`. Bài 2: ma trận `3 x 3` cho kết quả `1 x 1`: `max` toàn bộ = `9`.
- **Mức 2 (vừa):** Bài 3: đổi `max` thành `min`. Bài 4: `range(n - k + 1)` cho `i`, `j`; khối `range(i, i + k)`, `range(j, j + k)`.
- **Mức 3 (sâu):** Bài 5: duyệt độ lệch `-1..1`, kiểm tra `0 <= r < n` và `0 <= c < m` rồi cộng; chia nguyên `//` cho số ô hợp lệ.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [26]:
# Bài 1 và 2
assert list(zip([1, 2, 3, 4], [1, 2, 3, 4][1:], [1, 2, 3, 4][2:])) == [(1, 2, 3), (2, 3, 4)]
assert max(max([1, 5, 2]), max([4, 3])) == 5
assert Solution().largestLocal([[1, 2, 3], [4, 9, 6], [7, 8, 5]]) == [[9]]

# Bài 3
def smallest_local(grid):
    n = len(grid)
    return [[min(grid[r][c] for r in range(i, i + 3) for c in range(j, j + 3)) for j in range(n - 2)] for i in range(n - 2)]

# Bài 4
def largest_local_k(grid, k):
    n = len(grid)
    return [[max(grid[r][c] for r in range(i, i + k) for c in range(j, j + k)) for j in range(n - k + 1)] for i in range(n - k + 1)]

# Bài 5
def smoother(grid):
    rows, cols = len(grid), len(grid[0])
    out = []
    for i in range(rows):
        row = []
        for j in range(cols):
            total = count = 0
            for r in range(i - 1, i + 2):
                for c in range(j - 1, j + 2):
                    if 0 <= r < rows and 0 <= c < cols:
                        total += grid[r][c]
                        count += 1
            row.append(total // count)
        out.append(row)
    return out

assert smallest_local([[9, 9, 8, 1], [5, 6, 2, 6], [8, 2, 6, 4], [6, 2, 2, 2]]) == [[2, 1], [2, 2]]
assert smallest_local([[1, 2, 3], [4, 5, 6], [7, 8, 9]]) == [[1]]
assert largest_local_k([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 14, 15, 16]], 2) == [[6, 7, 8], [10, 11, 12], [14, 15, 16]]
assert largest_local_k([[1, 2], [3, 4]], 2) == [[4]]
assert largest_local_k([[9, 9, 8, 1], [5, 6, 2, 6], [8, 2, 6, 4], [6, 2, 2, 2]], 3) == [[9, 9], [8, 6]]
assert smoother([[1, 1, 1], [1, 0, 1], [1, 1, 1]]) == [[0, 0, 0], [0, 0, 0], [0, 0, 0]]
assert smoother([[100, 200, 100], [200, 50, 200], [100, 200, 100]]) == [[137, 141, 137], [141, 138, 141], [137, 141, 137]]
assert smoother([[5]]) == [[5]]
print("tất cả bài tập OK")

tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (cửa sổ `3 x 3` trượt, tính `max`)

```python
class Solution:
    def largestLocal(self, grid: List[List[int]]) -> List[List[int]]:
        n = len(grid)
        return [
            [max(grid[r][c] for r in range(i, i + 3) for c in range(j, j + 3)) for j in range(n - 2)]
            for i in range(n - 2)
        ]
```

**Checklist**

- [ ] Kết quả cỡ `(n - 2) x (n - 2)` (`range(n - 2)` cho cả `i` và `j`)?
- [ ] Khối là `range(i, i + 3)` x `range(j, j + 3)` (góc trên-trái), không có chỉ số âm?
- [ ] Không dùng `[[0] * m] * k` để tạo ma trận?
- [ ] Không sửa `grid` gốc?
- [ ] Không khởi tạo `best = 0` khi dữ liệu có thể âm (ở đề này `≥ 1` nên an toàn)?
- [ ] Đã thử `n = 3` (kết quả `1 x 1`) và `n = 4`?
- [ ] Signature đúng `largestLocal(self, grid: List[List[int]]) -> List[List[int]]` và có `from typing import List`?